# Train a Weapon-Detection YOLOv8 Model (Google Colab)

Fine-tune YOLOv8 on a gun/knife dataset using Colab's free GPU, then download `best.pt` for the Real-Time Weapon Detection app.

**Steps:** enable a GPU runtime (Runtime -> Change runtime type -> T4 GPU), then run the cells top to bottom.

You need a free Roboflow API key: https://app.roboflow.com -> Settings -> API.

## 1. Confirm a GPU is available

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow

## 3. Download a labelled weapon dataset from Roboflow Universe

Browse https://universe.roboflow.com and search "weapon detection" / "pistol" / "knife". Open a dataset, click **Download Dataset -> YOLOv8 -> show download code**, and copy the `workspace`, `project`, and `version` values into the cell below.

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "PASTE_YOUR_API_KEY"  # https://app.roboflow.com -> Settings -> API
WORKSPACE = "PASTE_WORKSPACE"
PROJECT = "PASTE_PROJECT"
VERSION = 1  # dataset version number

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
print("Dataset at:", dataset.location)

## 4. Train

50 epochs on a T4 GPU typically takes tens of minutes depending on dataset size. Start with `yolov8n.pt` (fast); use `yolov8s.pt`/`yolov8m.pt` for higher accuracy.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="weapon_yolov8",
)

## 5. Validate (precision / recall / mAP)

In [ ]:
metrics = model.val()
print(metrics)

## 6. Download best.pt

Save `best.pt` and copy it into your project (e.g. `models/best.pt`), then run:

```
weapon-detector --weights models/best.pt
```

or set `model.weights: models/best.pt` in your config.

In [ ]:
from google.colab import files

best = f"{results.save_dir}/weights/best.pt"
print("Best weights:", best)
files.download(best)